# Notebook 03 — Objetivo Específico 1: Análisis de Componentes Principales (PCA) y Selección de Bandas

- **Instancia SEPAL recomendada:** `m2` o `m4` (cómputo estadístico 100% local en Python en segundos).
- **Qué calcula:**
  1. Lee directamente `muestras_limpias_<region>_70_30_con_bandas.csv` generado en el Notebook 02 (**0% GEE**).
  2. Ejecuta PCA estandarizado sobre entrenamiento de **Mangle** (24 bandas biestacionales, 12 Seca, 12 Lluvia) bajo criterio de **Kaiser** ($\lambda \ge 1$).
  3. Cuantifica **contribución relativa de cada banda óptica y polarización SAR**, contraste Seca vs. Lluvia, y separabilidad multiclase supervisada (**ANOVA F** y **$d$ de Cohen** Mangle vs. Latifoliado).
  4. Exporta `OE1_PCA_Resultados_<region>.xlsx`, `bandas_seleccionadas_pca_<region>.json` y figuras a 300 DPI.

In [ ]:
# 1. Configuracion y carga de muestras ya extraidas en SEPAL (sin GEE)
import json
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "config.yaml").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

from src.config_utils import ensure_output_dirs, get_predictor_band_names, load_config
from src.plotting import (
    plot_band_contributions,
    plot_correlation_matrix,
    plot_scree_and_variance,
)

# >>> CAMBIAR AQUI ENTRE "PACIFICO" Y "CARIBE" <<<
REGION = "PACIFICO"

cfg = load_config()
dirs = ensure_output_dirs(cfg)
BANDS = get_predictor_band_names(cfg)
reg_l = REGION.lower()

csv_muestras = dirs["muestras"] / f"muestras_limpias_{reg_l}_70_30_con_bandas.csv"
assert csv_muestras.is_file(), f"No se encontro {csv_muestras}. Ejecuta primero el Notebook 02."
df_all = pd.read_csv(csv_muestras)

print(f"Muestras limpias cargadas: {len(df_all)} ({REGION})")
display(pd.crosstab(df_all["clase"], df_all["split"], margins=True))

In [ ]:
# 2. Motor matematico de PCA estandarizado (criterio de Kaiser lambda >= 1)
def ejecutar_pca_completo(df_subset: pd.DataFrame, columnas: list[str], kaiser_umbral: float = 1.0) -> dict:
    X = df_subset[columnas].dropna().values
    scaler = StandardScaler()
    X_std = scaler.fit_transform(X)

    cov_df = pd.DataFrame(np.cov(X_std, rowvar=False), index=columnas, columns=columnas)
    corr_df = pd.DataFrame(np.corrcoef(X_std, rowvar=False), index=columnas, columns=columnas)

    pca = PCA()
    scores = pca.fit_transform(X_std)
    eigenvalues = pca.explained_variance_
    var_ratio = pca.explained_variance_ratio_
    cum_var = np.cumsum(var_ratio)
    n_kaiser = max(1, int(np.sum(eigenvalues >= kaiser_umbral)))

    pc_names = [f"PC{i+1}" for i in range(len(columnas))]
    eigenvectors_df = pd.DataFrame(pca.components_.T, index=columnas, columns=pc_names)
    loadings_df = eigenvectors_df.multiply(np.sqrt(eigenvalues), axis=1)

    sq_evec = pca.components_[:n_kaiser, :].T ** 2
    pesos = var_ratio[:n_kaiser]
    contrib_raw = sq_evec @ pesos
    contrib_pct = 100.0 * contrib_raw / contrib_raw.sum()

    ranking_df = pd.DataFrame(
        {
            "contribucion_pca_pct": contrib_pct,
            "sensor": ["SAR" if (c.startswith("VV") or c.startswith("VH")) else "Optico_S2" for c in columnas],
            "temporada": ["SECA" if c.endswith("_SECA") else "LLUVIA" for c in columnas],
        },
        index=columnas,
    ).sort_values("contribucion_pca_pct", ascending=False)

    eigen_df = pd.DataFrame(
        {
            "Componente": pc_names,
            "Eigenvalor": eigenvalues,
            "Varianza_pct": var_ratio * 100.0,
            "Varianza_acum_pct": cum_var * 100.0,
            "Retenido_Kaiser": eigenvalues >= kaiser_umbral,
        }
    )

    return {
        "n_muestras": len(X),
        "n_kaiser": n_kaiser,
        "cov": cov_df,
        "corr": corr_df,
        "eigen": eigen_df,
        "eigenvectors": eigenvectors_df,
        "loadings": loadings_df,
        "ranking": ranking_df,
        "scores": scores,
    }

df_mangle_train = df_all[(df_all["split"] == "train") & (df_all["clase_id"] == 1)].copy()
print(f"Muestras de Mangle (train 70%) para PCA: {len(df_mangle_train)}")

kaiser = float(cfg["pca"]["criterio_kaiser"])
pca_bi = ejecutar_pca_completo(df_mangle_train, BANDS["todas"], kaiser_umbral=kaiser)
pca_seca = ejecutar_pca_completo(df_mangle_train, BANDS["seca_todas"], kaiser_umbral=kaiser)
pca_lluvia = ejecutar_pca_completo(df_mangle_train, BANDS["lluvia_todas"], kaiser_umbral=kaiser)

display(pca_bi["eigen"].head(8))
print(f"Componentes retenidos (Kaiser >= {kaiser}): {pca_bi['n_kaiser']} | Varianza acumulada: {pca_bi['eigen'].loc[pca_bi['n_kaiser']-1, 'Varianza_acum_pct']:.2f}%")

In [ ]:
# 3. Figuras oficiales del Objetivo 1 (300 DPI)
reg_nom = cfg["regiones"][REGION]["nombre_largo"]

plot_correlation_matrix(
    pca_bi["corr"],
    f"Matriz de Correlación de Pearson (24 Bandas S1+S2) — Mangle {reg_nom}",
    save_path=dirs["figuras"] / f"OE1_fig1_correlacion_24b_{reg_l}.png",
)
plt.show()

plot_scree_and_variance(
    pca_bi["eigen"]["Eigenvalor"].values,
    pca_bi["eigen"]["Varianza_pct"].values / 100.0,
    f"Análisis de Autovalores y Varianza Explicada (PCA Biestacional) — {reg_nom}",
    kaiser_threshold=kaiser,
    save_path=dirs["figuras"] / f"OE1_fig2_scree_varianza_{reg_l}.png",
)
plt.show()

plot_band_contributions(
    pca_bi["ranking"],
    "contribucion_pca_pct",
    f"Contribución Relativa de Bandas S2 y Polarizaciones SAR (PCA Retenido) — {reg_nom}",
    save_path=dirs["figuras"] / f"OE1_fig3_contribucion_bandas_{reg_l}.png",
)
plt.show()

In [ ]:
# 4. Contraste temporal (Seca vs Lluvia) + Separabilidad Multiclase (ANOVA F y d de Cohen)
comp_temp = pd.DataFrame([
    {"Analisis": "Biestacional (24 bandas)", "N_muestras": pca_bi["n_muestras"], "PCs_Kaiser": pca_bi["n_kaiser"], "Varianza_PC1_pct": pca_bi["eigen"].loc[0, "Varianza_pct"], "Varianza_Retenida_pct": pca_bi["eigen"].loc[pca_bi["n_kaiser"] - 1, "Varianza_acum_pct"]},
    {"Analisis": "Solo Epoca Seca (12 bandas)", "N_muestras": pca_seca["n_muestras"], "PCs_Kaiser": pca_seca["n_kaiser"], "Varianza_PC1_pct": pca_seca["eigen"].loc[0, "Varianza_pct"], "Varianza_Retenida_pct": pca_seca["eigen"].loc[pca_seca["n_kaiser"] - 1, "Varianza_acum_pct"]},
    {"Analisis": "Solo Epoca Lluviosa (12 bandas)", "N_muestras": pca_lluvia["n_muestras"], "PCs_Kaiser": pca_lluvia["n_kaiser"], "Varianza_PC1_pct": pca_lluvia["eigen"].loc[0, "Varianza_pct"], "Varianza_Retenida_pct": pca_lluvia["eigen"].loc[pca_lluvia["n_kaiser"] - 1, "Varianza_acum_pct"]},
])
display(comp_temp)

df_train = df_all[df_all["split"] == "train"].copy()
sep_rows = []
for b in BANDS["todas"]:
    grupos = [grp[b].values for _, grp in df_train.groupby("clase_id")]
    f_stat, _ = stats.f_oneway(*grupos)
    m_v = df_train.loc[df_train["clase_id"] == 1, b].values
    l_v = df_train.loc[df_train["clase_id"] == 2, b].values
    if len(l_v) > 1 and len(m_v) > 1:
        sp = np.sqrt(((len(m_v)-1)*np.var(m_v, ddof=1) + (len(l_v)-1)*np.var(l_v, ddof=1)) / (len(m_v)+len(l_v)-2))
        d_cohen = abs(np.mean(m_v) - np.mean(l_v)) / max(sp, 1e-9)
    else:
        d_cohen = np.nan
    sep_rows.append({
        "banda": b,
        "contribucion_pca_pct": pca_bi["ranking"].loc[b, "contribucion_pca_pct"],
        "ANOVA_F_5clases": f_stat,
        "Cohen_d_Mangle_vs_Latifoliado": d_cohen,
        "sensor": pca_bi["ranking"].loc[b, "sensor"],
        "temporada": pca_bi["ranking"].loc[b, "temporada"],
    })

df_sep = pd.DataFrame(sep_rows).set_index("banda").sort_values("contribucion_pca_pct", ascending=False)
display(df_sep)

In [ ]:
# 5. Seleccion de bandas optimas (OE1) y exportacion de Excel + JSON para Notebook 04
umbral_contrib = 100.0 / len(BANDS["todas"])
bandas_pca_sel = df_sep[df_sep["contribucion_pca_pct"] >= umbral_contrib].index.tolist()

sar_en_sel = [b for b in bandas_pca_sel if b.startswith(("VV", "VH"))]
if len(sar_en_sel) < 2:
    for b in df_sep[df_sep["sensor"] == "SAR"].head(2).index.tolist():
        if b not in bandas_pca_sel:
            bandas_pca_sel.append(b)

print(f"Umbral medio teorico (100/{len(BANDS['todas'])}): {umbral_contrib:.2f}%")
print(f"Bandas seleccionadas ({len(bandas_pca_sel)}):", bandas_pca_sel)

sel_payload = {
    "region": REGION,
    "criterio": f"contribucion_pca_pct >= {umbral_contrib:.2f}% + garantia multisensor",
    "n_componentes_kaiser": pca_bi["n_kaiser"],
    "varianza_retenida_pct": float(pca_bi["eigen"].loc[pca_bi["n_kaiser"] - 1, "Varianza_acum_pct"]),
    "bandas_seleccionadas": bandas_pca_sel,
    "bandas_opticas_sel": [b for b in bandas_pca_sel if not b.startswith(("VV", "VH"))],
    "bandas_sar_sel": [b for b in bandas_pca_sel if b.startswith(("VV", "VH"))],
}
json_path = dirs["tablas"] / f"bandas_seleccionadas_pca_{reg_l}.json"
json_path.write_text(json.dumps(sel_payload, indent=2, ensure_ascii=False), encoding="utf-8")

xlsx_path = dirs["tablas"] / f"OE1_PCA_Resultados_{reg_l}.xlsx"
with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
    comp_temp.to_excel(writer, sheet_name="Resumen_Temporal", index=False)
    pca_bi["eigen"].to_excel(writer, sheet_name="Eigenvalores_24b", index=False)
    df_sep.to_excel(writer, sheet_name="Ranking_PCA_y_Separab")
    pca_bi["loadings"].to_excel(writer, sheet_name="Loadings_24b")
    pca_bi["eigenvectors"].to_excel(writer, sheet_name="Eigenvectores_24b")
    pca_bi["corr"].to_excel(writer, sheet_name="Correlacion_24b")
    pca_bi["cov"].to_excel(writer, sheet_name="Covarianza_24b")

print(f"[OK] JSON guardado  : {json_path}")
print(f"[OK] Excel guardado : {xlsx_path}")